### Lab 6: Recurrent Neural Networks (RNN)

#### Objective
To understand the architecture and working mechanism of Recurrent Neural Networks (RNNs) by implementing and analyzing a simple sequence-prediction model.

#### Theory

Standard neural architectures excel at learning static input-output mappings. For instance, if a food stall's menu item were determined purely by the day's weather, a simple linear classifier would suffice—Sunny might map to Spring Rolls, Rainy to Noodles. However, decision-making in dynamic environments often involves history. What a vendor serves tomorrow may depend on what they served today, creating a sequence that unfolds over time.

When predictions are conditioned on past events, the model must possess a form of memory. Without it, each prediction is made in isolation, ignoring valuable contextual clues embedded in the sequence of previous states.

##### Introducing Recurrent Connections

Recurrent Neural Networks (RNNs) overcome this limitation by introducing a feedback loop. Instead of processing each input independently, an RNN maintains an internal state vector that is updated at every time step. This state acts as a compressed summary of all previous inputs in the sequence. The recurrent connection allows information to persist, effectively giving the network a short-term memory.

The unrolled view of an RNN reveals that it can be interpreted as a deep feedforward network where the weights are shared across all time steps. This weight sharing is what enables the network to generalise across sequences of varying lengths.

##### Core Mechanics: The Hidden State

At the heart of the RNN is the hidden state, denoted as $h_t$. At each time step $t$, the network takes two inputs:
1. The current external input $x_t$ (e.g., the current observation).
2. The previous hidden state $h_{t-1}$ (the memory from the past).

The new hidden state is computed by combining these two sources through a linear transformation followed by a non-linear activation function, typically $\tanh$. This operation fuses the present with the past. The updated hidden state then flows into the next time step and is also used to generate the current output.

##### Mathematical Definition

The update mechanism and output generation are formally defined as:

$$h_t = \tanh(W_{ih} x_t + W_{hh} h_{t-1})$$
$$y_t = W_{fc} h_t$$

Where:
- $x_t$ is the input vector at time $t$.
- $h_t$ is the hidden state vector (size $H$).
- $y_t$ is the output logit vector.
- $W_{ih}$, $W_{hh}$, and $W_{fc}$ are learnable weight matrices governing the input-to-hidden, hidden-to-hidden, and hidden-to-output transformations, respectively.

The matrix $W_{hh}$ is crucial—it encodes how past information influences the current state, making the network sensitive to order and temporal patterns.

##### Applying the RNN to a Multi-Input Sequence

In this lab, we address a specific sequence prediction problem involving two simultaneous features:
- **The current dish** (A, B, or C).
- **The current weather** (Sunny or Rainy).

The generative rule is simple yet requires context:
- If the weather is **Sunny**, the next dish remains the same.
- If the weather is **Rainy**, the next dish advances cyclically (A → B → C → A).

Notice that to know whether the dish should advance, the network must know the *current* dish. However, the current dish itself is the result of the previous step's prediction. This circular dependency makes the task ideal for an RNN, as the hidden state can seamlessly carry the identity of the current dish across time steps.

##### Input Representation

To feed this data into the network, we encode the categorical variables as one-hot vectors:
- Dish: 3-dimensional vector (one element for A, B, C).
- Weather: 2-dimensional vector (one element for Sunny, Rainy).

These are concatenated to form the final input vector $x_t$ of size $D = 5$. The output is a 3-dimensional logit vector, which is mapped to dish probabilities using a softmax function during inference.

#### Implementation

In [5]:
import random
import torch
import torch.nn as nn
import torch.optim as optim

# Dataset parameters
dishes = ['A', 'B', 'C']
weather_types = ['Sunny', 'Rainy']
dish_to_idx = {d: i for i, d in enumerate(dishes)}
weather_to_idx = {w: i for i, w in enumerate(weather_types)}

def next_dish(dish):
    if dish == 'A':
        return 'B'
    elif dish == 'B':
        return 'C'
    else:
        return 'A'

def generate_sequence(length=1000):
    current_dish = 'A'
    inputs = []
    targets = []
    for _ in range(length):
        weather = random.choice(weather_types)
        inputs.append((current_dish, weather))
        if weather == 'Sunny':
            new_dish = current_dish
        else:
            new_dish = next_dish(current_dish)
        targets.append(new_dish)
        current_dish = new_dish
    return inputs, targets

def encode_input(dish, weather):
    x = torch.zeros(5)
    x[dish_to_idx[dish]] = 1.0
    x[3 + weather_to_idx[weather]] = 1.0
    return x

# Generating and encoding data
inputs, targets = generate_sequence(length=2000)
X = torch.stack([encode_input(d, w) for d, w in inputs])
y = torch.tensor([dish_to_idx[t] for t in targets])

# Reshaping for RNN: (batch_size, seq_len, input_size)
X = X.unsqueeze(0)  # (1, 2000, 5)
y = y.unsqueeze(0)  # (1, 2000)

print(f"Input shape: {X.shape}")
print(f"Target shape: {y.shape}")

Input shape: torch.Size([1, 2000, 5])
Target shape: torch.Size([1, 2000])


##### Model Definition

In [6]:
class DishRNN(nn.Module):
    def __init__(self, input_size=5, hidden_size=3, output_size=3):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            bias=False,
            nonlinearity="tanh"
        )
        self.fc = nn.Linear(hidden_size, output_size, bias=False)

    def forward(self, x):
        rnn_out, hidden = self.rnn(x)
        logits = self.fc(rnn_out)
        return logits

model = DishRNN()
print(model)

DishRNN(
  (rnn): RNN(5, 3, bias=False, batch_first=True)
  (fc): Linear(in_features=3, out_features=3, bias=False)
)


#### Training

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 300
for epoch in range(epochs):
    optimizer.zero_grad()
    logits = model(X)
    loss = criterion(logits.view(-1, 3), y.view(-1))
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] Loss = {loss.item():.6f}")

Epoch [50/300] Loss = 0.841485
Epoch [100/300] Loss = 0.368574
Epoch [150/300] Loss = 0.116750
Epoch [200/300] Loss = 0.059880
Epoch [250/300] Loss = 0.038331
Epoch [300/300] Loss = 0.027282


##### Results

##### Training Performance

The model was trained for 300 epochs. The loss converged successfully, indicating the model learned the underlying sequence patterns.

##### Model Accuracy

with torch.no_grad():
    logits = model(X)
    preds = logits.argmax(dim=-1)
    accuracy = (preds == y).float().mean().item()
print(f"Training Accuracy: {accuracy * 100:.2f}%")

In [8]:
### Learned Transitions

print("\nLearned transition rules:\n")
test_cases = [
    ('A', 'Sunny'), ('A', 'Rainy'),
    ('B', 'Sunny'), ('B', 'Rainy'),
    ('C', 'Sunny'), ('C', 'Rainy'),
]
for dish, weather in test_cases:
    x = encode_input(dish, weather).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        logits = model(x)
        pred = logits.argmax(-1).item()
    print(f"Current Dish={dish}, Weather={weather:5s} → Predicted Next Dish={dishes[pred]}")


Learned transition rules:

Current Dish=A, Weather=Sunny → Predicted Next Dish=A
Current Dish=A, Weather=Rainy → Predicted Next Dish=B
Current Dish=B, Weather=Sunny → Predicted Next Dish=B
Current Dish=B, Weather=Rainy → Predicted Next Dish=C
Current Dish=C, Weather=Sunny → Predicted Next Dish=C
Current Dish=C, Weather=Rainy → Predicted Next Dish=A


#### Discussion
In this lab, architecture and working mechanism of Recurrent Neural Networks (RNNs) by implementing and analyzing a simple sequence-prediction model. we first learned about RNN, we performed the code implementation by first defining our model and then training it. In training, the model was trained for 300 ephocs and it's losses showed smooth convergance.The absence of vanishing gradients here is due to shallow unrolling and the use of `tanh` activation, which kept values bounded. we also worked on accuracy of model and it achieved 100% accuracy.Unlike a feedforward network, which would treat each time step independently, the RNN leverages its hidden state to carry context from one step to the next. This is evident because the decision “rainy → next dish” requires knowing the current dish, which is itself the output of the previous step, a perfect use case for recurrence.

What `hidden_size` mean for RNN.

The `hidden_size` (H) defines the dimensionality of the recurrent state vector that an RNN passes forward through time. It is the fundamental measure of the network's memory capacity. Technically, `hidden_size` determines the dimensions of all weight matrices, most critically the hidden‑to‑hidden matrix (H×H), which grows quadratically with H. A larger H enables the network to encode more complex and longer‑range temporal patterns, but this extra representational power comes at the cost of a steep increase in parameters and a higher risk of overfitting, making the choice of H a critical trade‑off between expressive capacity and generalisation performance.

#### Conclusion
In this lab exercise we sucsessfully implemented RNN and learned about it's architecture and working mechenisms. The model demonstrated how recurrent connections allow past information to persist, enabling genuinely context‑aware predictions. Achieving perfect accuracy validated the working mechenisms of model. The choice of `hidden_size` proved crucial, as it defined the memory capacity and directly scaled the parameter count. A value of three perfectly matched the task's three discrete states, avoiding both underfitting and unnecessary over‑parameterisation. This trade‑off highlights that the `hidden_size` must always be carefully calibrated to the complexity of the sequence dynamics. While a basic RNN handled this simple cycle effectively, more elaborate architectures would be necessary for longer or noisier sequences in real‑world applications.

